In [1]:
import numpy as np
from astropy.nddata import block_replicate, block_reduce

In [ ]:
def upscale(
    data: np.array,
    upscale_y: int = 1,
    upscale_x: int = 1,
) -> np.array:
    """
    Upscale a 2D array by repeating elements along each axis.

    Args:
        data (np.array): Input 2D array.
        upscale_y (int): Upscaling factor over the y direction.
        upscale_x (int): Upscaling factor over the x direction.

    Returns:
        output (np.array): Oversampled array.

    Raises:
        ValueError: if upscale factors are not positive integers.
    
    Notes:
        - The array total sum is conserved through linear interpolation.
        - For N-dim arrays, consider using `astropy.nndata.block_replicate()`.
    """
    if not (
        (isinstance(upscale_y, int) and upscale_y > 0) and
        (isinstance(upscale_x, int) and upscale_x > 0)
    ):
        raise ValueError("Upscaling factors must be positive integers.")
    
    for i, f in enumerate((upscale_y, upscale_x)):
        data = np.repeat(data, f, axis=i)
    
    return data/np.prod((upscale_y, upscale_x))


def downscale(
    data: np.array,
    downscale_y: int = 1,
    downscale_x: int = 1,
) -> np.array:
    """
    Downscale a 2D array.

    Args:
        data (np.array): Input 2D array.
        downscale_y (int): Downscaling factor over the y direction.
        downscale_x (int): Downscaling factor over the x direction.

    Returns:
        output (np.array): Downsampled array.

    Raises:
        ValueError: if downscale factors are not positive integers.
    
    Notes:
        - The downsampling is performed through blocks subdivision, which
          represent the elements of the downsampled array. Each block is
          reduced by adding its elements for linear interpolation.
        - The total sum of the array is conserved.
        - For N-dim arrays, consider using `astropy.nndata.block_reduce()`.
        
        TODO: extract values in lost cols/rows when `data.shape` is not
              evenly divisible by the downscale factors and add them to
              the input array axis (so that the array sum is conserved).
    """
    def _handle_shape(
        data: np.array,
        downscaling: np.array,
    ) -> np.array:
        """Adjusts array for blocks subdivision by cutting extra-rows/columns."""
        def _handle_axis(a: np.array, idx: int) -> np.array:
            """Redistributes cutted values in the block-adjusted axis."""
            return a[:idx] + a[idx:].sum(axis=0) / idx
        adj_shape = (np.array(data.shape) // downscaling) * downscaling
        for ax in range(data.ndim):
            if data.shape[ax] != adj_shape[ax]:
                data = data.swapaxes(0, ax)
                data = _handle_axis(data, adj_shape[ax])
                data = data.swapaxes(0, ax)
        return data

    def _to_blocks(
        data: np.array,
        downscaling: np.array,
    ) -> np.array:
        """Reshapes input array into blocks."""
        assert not np.any(np.mod(data.shape, downscaling) != 0)
        nblocks = np.array(data.shape) // downscaling
        reshaping = tuple(dim for dims in zip(nblocks, downscaling) for dim in dims)
        return data.reshape(reshaping).transpose((0, 2, 1, 3))
    
    if not (
        (isinstance(downscale_y, int) and downscale_y > 0) and
        (isinstance(downscale_x, int) and downscale_x > 0)
    ):
        raise ValueError("Downscaling factors must be positive integers.")

    downscaling = np.array((downscale_y, downscale_x))
    data = _handle_shape(data, downscaling)
    data = _to_blocks(data, downscaling)
    return data.sum(axis=(2, 3))


# TODO:
#   1. update bloodmoon w `upscale()` and `downscale()`
#   2. write tests for them
#
#   3. update scripts that use upscaling (e.g. IROS_pipeline.py) bc the input are swapped
#   4. same reason, update `demo.ipynb` for bloodmoon

In [3]:
import mbloodmoon as bm
from IROS_pipeline import _handle_dirpaths

mask_FITS = "wfm_mask.fits"

skyfield = "GalacticCenter"
data_FITS = "20241011_galctr_rxte_sax_2-30keV_1ks_2cams_sources_cxb"

mask_file, simul_data, save_path = _handle_dirpaths(
    mask=mask_FITS,
    skyfield=skyfield,
    simul=data_FITS,
)

wfm = bm.codedmask(mask_file, upscale_x=1, upscale_y=1)

In [6]:
sky = np.ones(wfm.sky_shape)

down_sky = downscale(sky, *(5, 3))
up_sky = upscale(down_sky, *(5, 3))


sky.shape, up_sky.shape, down_sky.shape, int(sky.sum()), int(up_sky.sum()), int(down_sky.sum())

((1033, 1671), (1030, 1671), (206, 557), 1726143, 1726143, 1726143)

In [15]:
wfm2 = bm.codedmask(mask_file, upscale_x=3, upscale_y=5)

wfm2.sky_shape

(5163, 5015)

In [ ]:
def downscale(
    data: np.array,
    downscale_y: int = 1,
    downscale_x: int = 1,
) -> np.array:
    """Downscale a 2D array."""    
    def _handle_shape(
        data: np.array,
        downscaling: np.array,
    ) -> np.array:
        """Adjusts array for blocks subdivision by cutting extra-rows/columns."""
        def _handle_axis(a: np.array, idx: int) -> np.array:
            """Redistributes cutted values in the block-adjusted axis."""
            return a[:idx] + a[idx:].sum(axis=0) / idx
        
        adj_shape = (np.array(data.shape) // downscaling) * downscaling
        for ax in range(data.ndim):
            if data.shape[ax] != adj_shape[ax]:
                data = data.swapaxes(0, ax)
                data = _handle_axis(data, adj_shape[ax])
                data = data.swapaxes(0, ax)
        return data

    def _to_blocks(
        data: np.array,
        downscaling: np.array,
    ) -> np.array:
        """Reshapes input array into blocks."""
        assert not np.any(np.mod(data.shape, downscaling) != 0)
        nblocks = np.array(data.shape) // downscaling
        reshaping = tuple(dim for dims in zip(nblocks, downscaling) for dim in dims)
        return data.reshape(reshaping).transpose((0, 2, 1, 3))
    
    if not (
        (isinstance(downscale_y, int) and downscale_y > 0) and
        (isinstance(downscale_x, int) and downscale_x > 0)
    ):
        raise ValueError("Downscaling factors must be positive integers.")

    downscaling = np.array((downscale_y, downscale_x))
    data = _handle_shape(data, downscaling)
    data = _to_blocks(data, downscaling)
    return data.sum(axis=(2, 3))









a = np.random.uniform(0, 10, (12, 15))
fy, fx = 5, 4
reduced_a = block_reduce(a, (fy, fx))
downsampled_a = downscale(a, *(fy, fx))


print(
    np.all(downsampled_a == reduced_a),
    a.shape,
    reduced_a.shape,
    downsampled_a.shape,
)

np.sum(a), np.sum(downsampled_a), np.sum(reduced_a)

(10, 15)
(10, 12)
False (12, 15) (2, 3) (2, 3)


(np.float64(884.7817808357865),
 np.float64(884.7817808357864),
 np.float64(586.6241431401899))